# Multi-species LSTM vs. single-species LSTMs — broad species mix

**Question.** Do LSTMs trained on pooled multi-species data with a species
context vector (one-hot or phylogenetic) outperform single-species LSTMs
trained on one species at a time — *especially* for species whose own
dataset is too small to train a reliable LSTM on its own?

**Species.**

| Region | Dataset | Target key | Species |
|---|---|---|---|
| Japan | `GMU_Cherry_Japan` | `gmu_0` | *Prunus yedoensis* |
| Switzerland | `GMU_Cherry_Switzerland` | `gmu_1` | *Prunus avium* |
| S. Korea | `GMU_Cherry_South_Korea` | `gmu_2` | *Prunus yedoensis* |
| W. Europe | `PEP725_Apple` | `BBCH_60` | *Malus × domestica* |
| W. Europe | `PEP725_Pear` | `BBCH_60` | *Pyrus communis* |
| W. Europe | `PEP725_Peach` | `BBCH_60` | *Prunus persica* |
| W. Europe | `PEP725_Almond` | `BBCH_60` | *Prunus amygdalis* |
| W. Europe | `PEP725_Cherry` | `BBCH_60` | *Prunus avium* |
| W. Europe | `PEP725_Apricot` | `BBCH_60` | *Prunus armeniaca* |
| W. Europe | `PEP725_Plum` | `BBCH_60` | *Prunus domestica* |

**Splits.** Temporal: years `< CUTOFF` for training, years `≥ CUTOFF` for
test. The multi-species training set is the union of the per-species
training splits; every model is evaluated on the *same* per-species test
splits so numbers are directly comparable.

**Small-sample regime.** A single-species LSTM is only trained when the
dataset has at least `MIN_SAMPLES_FOR_SINGLE` training samples. Species
below this threshold are *still* evaluated on the multi-species models, so
you can see how well the context-aware pooled models compensate for small
per-species data.

**Caching.** Every trained model is saved to
`outputs/models/<name>/<name>.pt` and auto-loaded on subsequent runs.
Set `RETRAIN = True` to force re-training.

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
RETRAIN = False                     # True → retrain every model
CUTOFF  = 2006                      # years < CUTOFF → train, ≥ CUTOFF → test
MIN_SAMPLES_FOR_SINGLE = 100        # skip single-species LSTM below this

DATASETS_CONFIG = {
    # label                 registry key              target key         scientific name
    'GMU_Japan':    dict(key='GMU_Cherry_Japan_Y',     obs_key='gmu_0',   name='Prunus yedoensis'),
    'GMU_Swiss':    dict(key='GMU_Cherry_Switzerland', obs_key='gmu_1',   name='Prunus avium'),
    'GMU_SKorea':   dict(key='GMU_Cherry_South_Korea', obs_key='gmu_2',   name='Prunus yedoensis'),
    'Apple':        dict(key='PEP725_Apple',           obs_key='BBCH_60', name='Malus x domestica'),
    'Pear':         dict(key='PEP725_Pear',            obs_key='BBCH_60', name='Pyrus communis'),
    'Peach':        dict(key='PEP725_Peach',           obs_key='BBCH_60', name='Prunus persica'),
    'Almond':       dict(key='PEP725_Almond',          obs_key='BBCH_60', name='Prunus amygdalis'),
    'Cherry':       dict(key='PEP725_Cherry',          obs_key='BBCH_60', name='Prunus avium'),
    'Apricot':      dict(key='PEP725_Apricot',         obs_key='BBCH_60', name='Prunus armeniaca'),
    'Plum':         dict(key='PEP725_Plum',            obs_key='BBCH_60', name='Prunus domestica'),
}

DATA_KEYS = ['temperature_2m_mean', 'daylight_duration']
PHYLO_K_EMBED = 8

MODEL_KWARGS = dict(
    data_keys   = DATA_KEYS,
    hidden_size = 64,
    num_layers  = 2,
)
TRAIN_KWARGS = dict(
    num_epochs               = 1000,
    batch_size               = 512,
    val_period               = 10,
    optimizer                = 'adam',
    optimizer_kwargs         = dict(lr=1e-3, weight_decay=1e-4),
    scheduler_step_size      = 100,
    scheduler_decay          = 0.9,
    early_stopping           = True,
    early_stopping_patience  = 10,
    early_stopping_min_delta = 1e-3,
    early_stopping_rerun     = False,   # skipped — doubles training time
    seed                     = 0,
    verbose                  = True,
)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pysephone.dataset.dataset import Dataset
from pysephone.dataset.observations import Observations
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.dataset.util.phylogeny import PhylogenyFeatures
from pysephone.models.lstm import LSTMModel
from pysephone.models.lstm_ctx import (
    OneHotSpeciesLSTMModel,
    PhylogeneticLSTMModel,
)
from pysephone.paths import get_data_root, get_model_dir

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 1. Load every dataset

GMU sources don't ship `species_names`; we inject them from
`DATASETS_CONFIG` so the phylogeny fitter can resolve them. PEP725
sources already carry their scientific names.

A **universal target function** picks the right observation key from any
sample regardless of source (each sample has exactly one of
`BBCH_60`, `gmu_0`, `gmu_1`, `gmu_2`).

In [ ]:
_KNOWN_TARGET_KEYS = ('BBCH_60', 'gmu_0', 'gmu_1', 'gmu_2')

def target_fn(sample):
    obs = sample['observations']
    for k in _KNOWN_TARGET_KEYS:
        if k in obs:
            return obs[k]
    raise KeyError(f'no known target key in sample: {list(obs.keys())}')


def patch_species_names(obs: Observations, scientific_name: str) -> None:
    """Fill in species_name for every (src, species_id) that doesn't have
    one already. Safe to call on sources that ship names — existing entries
    are left untouched."""
    for ix in obs.iter_index():
        key = (ix[0], int(ix[3]))
        if key not in obs._species_names:
            obs._species_names[key] = scientific_name


cal   = Calendar()
feats = OpenMeteoFeatures(calendar=cal, data_keys=DATA_KEYS)

per_species_trn: dict[str, Dataset] = {}
per_species_tst: dict[str, Dataset] = {}

for label, cfg in DATASETS_CONFIG.items():
    ds = Dataset.load(cfg['key'], calendar=cal, feature_providers=[feats])
    patch_species_names(ds._obs, cfg['name'])
    ds.download_features(verbose=True)
    ys_trn = [y for y in ds.years if y <  CUTOFF]
    ys_tst = [y for y in ds.years if y >= CUTOFF]
    per_species_trn[label] = ds.select_years(ys_trn)
    per_species_tst[label] = ds.select_years(ys_tst)

In [ ]:
# Sample-count summary — drives the single-species LSTM go/no-go decision
rows = []
for label in DATASETS_CONFIG:
    rows.append({
        'label': label,
        'registry':      DATASETS_CONFIG[label]['key'],
        'train_n':       len(per_species_trn[label]),
        'test_n':        len(per_species_tst[label]),
        'single_LSTM?':  'yes' if len(per_species_trn[label]) >= MIN_SAMPLES_FOR_SINGLE else 'SKIP (too few)',
    })
display(pd.DataFrame(rows).set_index('label'))

SPECIES_FOR_SINGLE = [l for l, c in DATASETS_CONFIG.items()
                      if len(per_species_trn[l]) >= MIN_SAMPLES_FOR_SINGLE]
print(f'\nWill train single-species LSTMs for: {SPECIES_FOR_SINGLE}')
print(f'Too small for a single-species LSTM:   '
      f'{[l for l in DATASETS_CONFIG if l not in SPECIES_FOR_SINGLE]}')

In [ ]:
# Pooled training set: union of every per-species training split
pooled_obs_trn = reduce(
    Observations.merge,
    [per_species_trn[l]._obs for l in DATASETS_CONFIG],
)
ds_pool_trn = Dataset(pooled_obs_trn, calendar=cal, feature_providers=[feats])
ds_pool_trn.preload_features(verbose=True)

print(f'Pooled training set: {len(ds_pool_trn)} samples, '
      f'{len(pooled_obs_trn.species_names)} (src, species_id) pairs known')
print('Species names present for phylogeny:')
for k, v in sorted(pooled_obs_trn.species_names.items()):
    print(f'  {k}  →  {v}')

## 2. Fit the phylogenetic embedding (one OpenTree call)

In [ ]:
phylo = PhylogenyFeatures(k_embed=PHYLO_K_EMBED, output=['mds']).fit(pooled_obs_trn)
species_keys = list(phylo.species_keys)
print(f'species_keys  ({len(species_keys)} pairs):')
for k in species_keys:
    print(f'  {k}')
print(f'mds_coords shape: {phylo.mds_coords.shape}')

## 3. Train / load helper

Every model is saved to `outputs/models/<name>/<name>.pt`. A later run
reloads instantly unless `RETRAIN=True`.

In [ ]:
def train_or_load(model_name, model_cls, model_kwargs, dataset, retrain=False):
    path = get_model_dir(get_data_root(), model_name) / f'{model_name}.pt'
    if not retrain and path.exists():
        model = torch.load(path, weights_only=False, map_location=DEVICE)
        model.eval()
        print(f'[loaded]   {model_name}')
        return model, None

    print(f'[training] {model_name}  (n={len(dataset)}) ...')
    model, info = model_cls.fit(
        target_fn    = target_fn,
        dataset      = dataset,
        model_kwargs = model_kwargs,
        device       = DEVICE,
        **TRAIN_KWARGS,
    )
    model.save(model_name)
    model.eval()
    print(f'[saved]    {model_name}')
    return model, info

## 4. Train the ensemble

| name | class | training data | context |
|---|---|---|---|
| `lstm_single_<label>` | `LSTMModel` | that species only | — |
| `lstm_pool_plain` | `LSTMModel` | pooled (10 datasets) | — |
| `lstm_pool_onehot` | `OneHotSpeciesLSTMModel` | pooled | one-hot |
| `lstm_pool_phylo` | `PhylogeneticLSTMModel` | pooled | phylogeny MDS |

In [ ]:
models: dict = {}

# 4a. Single-species LSTMs — only where we have enough training samples
for label in SPECIES_FOR_SINGLE:
    name = f'lstm_single_{label}'
    models[name], _ = train_or_load(
        model_name   = name,
        model_cls    = LSTMModel,
        model_kwargs = dict(MODEL_KWARGS),
        dataset      = per_species_trn[label],
        retrain      = RETRAIN,
    )

In [ ]:
# 4b. Pooled LSTM, no species info  (sanity baseline)
models['lstm_pool_plain'], _ = train_or_load(
    model_name   = 'lstm_pool_plain',
    model_cls    = LSTMModel,
    model_kwargs = dict(MODEL_KWARGS),
    dataset      = ds_pool_trn,
    retrain      = RETRAIN,
)

# 4c. Pooled LSTM + one-hot species context
models['lstm_pool_onehot'], _ = train_or_load(
    model_name   = 'lstm_pool_onehot',
    model_cls    = OneHotSpeciesLSTMModel,
    model_kwargs = dict(MODEL_KWARGS, species_keys=species_keys),
    dataset      = ds_pool_trn,
    retrain      = RETRAIN,
)

# 4d. Pooled LSTM + phylogenetic MDS context
models['lstm_pool_phylo'], _ = train_or_load(
    model_name   = 'lstm_pool_phylo',
    model_cls    = PhylogeneticLSTMModel,
    model_kwargs = dict(MODEL_KWARGS,
                        species_keys=species_keys,
                        mds_coords=phylo.mds_coords),
    dataset      = ds_pool_trn,
    retrain      = RETRAIN,
)

## 5. Evaluate every model on every per-species test split

Single-species models are only evaluated on their own species. The three
pooled models are evaluated on **all** species — including those whose own
dataset was too small to train a single-species LSTM on.

In [ ]:
def evaluate(model, dataset, target_fn):
    errors, ys_true, ys_pred = [], [], []
    for sample in dataset.iter_items():
        try:
            target_date = target_fn(sample)
        except KeyError:
            continue
        _, info = model.predict(sample)
        t_ix = int((np.datetime64(target_date, 'D') -
                    np.datetime64(sample['season_start'], 'D')) / np.timedelta64(1, 'D'))
        p_ix = int(info['ix'])
        ys_true.append(t_ix); ys_pred.append(p_ix); errors.append(p_ix - t_ix)
    if not errors:
        return None
    e = np.array(errors, dtype=float)
    return {
        'n':    len(e),
        'MAE':  float(np.abs(e).mean()),
        'RMSE': float(np.sqrt((e**2).mean())),
        'bias': float(e.mean()),
        'ys_true': np.array(ys_true), 'ys_pred': np.array(ys_pred),
    }


rows = []
eval_cache: dict = {}
for model_name, model in models.items():
    for species_label, ds_tst in per_species_tst.items():
        if model_name.startswith('lstm_single_') and model_name != f'lstm_single_{species_label}':
            continue
        if len(ds_tst) == 0:
            continue
        m = evaluate(model, ds_tst, target_fn)
        if m is None:
            continue
        eval_cache[(model_name, species_label)] = m
        rows.append({
            'model':   model_name,
            'species': species_label,
            'n':       m['n'],
            'MAE':     round(m['MAE'],  2),
            'RMSE':    round(m['RMSE'], 2),
            'bias':    round(m['bias'], 2),
        })
df = pd.DataFrame(rows)
display(df)

In [ ]:
# Per-species MAE — one column per model, one row per species
pivot = df.pivot(index='species', columns='model', values='MAE')
# Preserve a sensible column order
cols_order = [f'lstm_single_{s}' for s in DATASETS_CONFIG] + \
             ['lstm_pool_plain', 'lstm_pool_onehot', 'lstm_pool_phylo']
pivot = pivot.reindex(
    columns=[c for c in cols_order if c in pivot.columns],
    index=list(DATASETS_CONFIG.keys()),
)
display(pivot.round(2))

# Improvement of each pooled model over the species' own single-species LSTM
print('\nΔ MAE vs. single-species LSTM (negative = pooled model is better).')
print('Rows for species without a single-species model show the pooled MAE absolute.')
summary_rows = []
for s in DATASETS_CONFIG:
    single_col = f'lstm_single_{s}'
    has_single = single_col in pivot.columns and not np.isnan(pivot.loc[s, single_col])
    row = {
        'species':    s,
        'train_n':    len(per_species_trn[s]),
        'single_MAE': round(float(pivot.loc[s, single_col]), 2) if has_single else '— (skipped)',
    }
    for pool in ['lstm_pool_plain', 'lstm_pool_onehot', 'lstm_pool_phylo']:
        if pool not in pivot.columns or np.isnan(pivot.loc[s, pool]):
            row[pool] = np.nan
            continue
        pool_mae = float(pivot.loc[s, pool])
        if has_single:
            row[pool] = f'{pool_mae:.2f}  (Δ {pool_mae - float(pivot.loc[s, single_col]):+.2f})'
        else:
            row[pool] = f'{pool_mae:.2f}'
    summary_rows.append(row)
display(pd.DataFrame(summary_rows).set_index('species'))

## 6. Visualisations

In [ ]:
# ── Bar chart: per-species MAE by model ──────────────────────────────────
# One group of bars per species, one bar per pooled model + the species'
# single-species LSTM (if present). Species with no single-LSTM clearly
# show pooled-only bars.

POOL_ORDER  = ['lstm_pool_plain', 'lstm_pool_onehot', 'lstm_pool_phylo']
POOL_COLORS = {'lstm_pool_plain': '#7f8c8d',
               'lstm_pool_onehot': '#2980b9',
               'lstm_pool_phylo':  '#27ae60'}

labels = list(DATASETS_CONFIG.keys())
n_species = len(labels)
bar_names = POOL_ORDER + ['lstm_single_{species}']   # last is placeholder
n_bars = len(bar_names)

fig, ax = plt.subplots(figsize=(12.5, 5))
x = np.arange(n_species)
w = 0.8 / n_bars

# Pooled models
for i, name in enumerate(POOL_ORDER):
    if name not in pivot.columns:
        continue
    vals = pivot[name].reindex(labels).values
    ax.bar(x + i * w - (n_bars - 1) * w / 2, vals, width=w * 0.95,
           color=POOL_COLORS[name], edgecolor='white', linewidth=0.5, label=name)

# Single-species bars (one color, grouped as last position)
single_vals = [pivot.loc[s, f'lstm_single_{s}']
               if f'lstm_single_{s}' in pivot.columns else np.nan
               for s in labels]
single_mask = [not np.isnan(v) for v in single_vals]
ax.bar(x[single_mask] + (n_bars - 1) * w - (n_bars - 1) * w / 2,
       [v for v, m in zip(single_vals, single_mask) if m],
       width=w * 0.95, color='#c0392b', edgecolor='white', linewidth=0.5,
       label='lstm_single_<species>')

# Mark species without a single-species LSTM
skipped = [i for i, m in enumerate(single_mask) if not m]
for i in skipped:
    ax.text(x[i] + (n_bars - 1) * w - (n_bars - 1) * w / 2, 0.3,
            'single LSTM\nskipped', ha='center', va='bottom',
            fontsize=7, color='grey', style='italic')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_ylabel('Test MAE (days)')
ax.set_title('Per-species test MAE — single vs. pooled / one-hot / phylogenetic LSTM',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ── MAE vs training-set size — does species-context matter more for small datasets? ─
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xscale('log')

sizes = np.array([len(per_species_trn[s]) for s in labels])
for name, color in POOL_COLORS.items():
    if name not in pivot.columns:
        continue
    maes = pivot[name].reindex(labels).values
    ax.plot(sizes, maes, marker='o', lw=1.5, color=color, label=name, alpha=0.9)

# Single LSTM only where it exists
mask = [f'lstm_single_{s}' in pivot.columns for s in labels]
s_sizes = sizes[mask]
s_maes  = [float(pivot.loc[s, f'lstm_single_{s}']) for s, m in zip(labels, mask) if m]
ax.plot(s_sizes, s_maes, marker='s', lw=1.5, color='#c0392b',
        label='lstm_single_<species>', alpha=0.9)

# Label each point with species name
for sp, size in zip(labels, sizes):
    if 'lstm_pool_phylo' in pivot.columns:
        y = pivot.loc[sp, 'lstm_pool_phylo']
        if not np.isnan(y):
            ax.annotate(sp, (size, y), fontsize=7, alpha=0.7,
                        xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Training-set size (samples, log scale)')
ax.set_ylabel('Test MAE (days)')
ax.set_title('How does model performance scale with per-species training size?',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True, which='both', ls=':', lw=0.5, alpha=0.5)
plt.tight_layout(); plt.show()

## 7. Interpretation checklist

- **Pooled beats single (all species)** → shared weather dynamics dominate; more
  data helps even without species labels.
- **Pooled beats single more for small species** (visible in the log-scale
  plot above) → pooled training rescues species with sparse data, which is
  the regime you asked about.
- **One-hot ≪ plain** → the model *can* use species identity when it's given.
- **Phylo ≈ one-hot** → relatedness adds nothing beyond identity in this
  closed set.  Phylo usually only wins when test species are rare (or absent)
  in training — consider a leave-one-species-out variant to probe that.
- **Phylo < one-hot for small species** → transferable structure kicks in
  exactly where one-hot can't generalise.